# 03. ML for Forecasting: Gradient Boosting with Lag Features

## Background

Gradient boosting machines (GBMs) — XGBoost, LightGBM, CatBoost — dominate structured data competition results and have been successfully applied to time series forecasting through careful feature engineering. The key insight: reframe forecasting as supervised learning by creating lag features (previous observations as input features) and calendar features (hour, day-of-week, month, holiday indicators).

This approach has significant advantages: GBMs naturally handle missing values, work well with static covariates (store size, product category), and are straightforward to interpret through SHAP values. The M5 competition (2020) — the most prestigious time series competition, predicting Walmart sales — was won by an ensemble of GBMs and neural networks, not by pure deep learning.

## What You'll Learn

- Lag feature engineering: which lags to include and why
- Calendar features: encoding cyclical time information
- Rolling statistical features: rolling mean, std, quantiles
- Direct vs recursive forecasting strategies

## Why This Matters

GBM forecasters are the practical workhorses of many production forecasting systems. They're interpretable, fast, robust to distribution shift in covariates, and easily extended with domain knowledge features. Understanding when to use GBMs vs deep learning is a key architectural judgment call for forecasting systems.


## Feature Engineering Pipeline

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from typing import List

np.random.seed(42)

def generate_ts(n=500):
    t = np.arange(n)
    values = 100 + 0.05*t + 10*np.sin(2*np.pi*t/52) + np.random.normal(0, 3, n)
    return pd.Series(values, index=pd.date_range('2018-01-01', periods=n, freq='W'), name='value')

ts = generate_ts()

def create_lag_features(series: pd.Series,
                          lags: List[int],
                          rolling_windows: List[int] = None) -> pd.DataFrame:
    df = pd.DataFrame({'value': series})

    # Lag features
    for lag in lags:
        df[f'lag_{lag}'] = series.shift(lag)

    # Rolling statistics
    if rolling_windows:
        for window in rolling_windows:
            df[f'roll_mean_{window}'] = series.shift(1).rolling(window).mean()
            df[f'roll_std_{window}'] = series.shift(1).rolling(window).std()
            df[f'roll_min_{window}'] = series.shift(1).rolling(window).min()
            df[f'roll_max_{window}'] = series.shift(1).rolling(window).max()

    # Calendar features
    df['week_of_year'] = series.index.isocalendar().week.astype(int)
    df['month'] = series.index.month
    df['quarter'] = series.index.quarter
    df['year'] = series.index.year

    # Cyclical encoding (avoids discontinuity at 52->1)
    df['week_sin'] = np.sin(2 * np.pi * df['week_of_year'] / 52)
    df['week_cos'] = np.cos(2 * np.pi * df['week_of_year'] / 52)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    return df

# Key lags: 1-week, 4-week, 52-week (same week last year), and nearby seasonal
LAGS = [1, 2, 3, 4, 8, 13, 26, 52, 53, 54]
ROLLING = [4, 8, 13, 26]

df = create_lag_features(ts, lags=LAGS, rolling_windows=ROLLING)
df.dropna(inplace=True)

print(f'Feature matrix: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'\nFeature groups:')
print(f'  Lag features:     {sum(1 for c in df.columns if c.startswith("lag_"))}')
print(f'  Rolling features: {sum(1 for c in df.columns if c.startswith("roll_"))}')
print(f'  Calendar:         {sum(1 for c in df.columns if c in ["week_of_year","month","quarter","year","week_sin","week_cos","month_sin","month_cos"])}')
print(f'\nFeature names: {list(df.columns[:8])}...')


## LightGBM Forecaster

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_percentage_error as sklearn_mape

# Time-based train/test split
HORIZON = 52
X = df.drop('value', axis=1)
y = df['value']

X_train = X.iloc[:-HORIZON]
y_train = y.iloc[:-HORIZON]
X_test = X.iloc[-HORIZON:]
y_test = y.iloc[-HORIZON:]

# Fit LightGBM
lgbm = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=10,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)
lgbm.fit(X_train, y_train, eval_set=[(X_test, y_test)], callbacks=[])

y_pred = lgbm.predict(X_test)
mape_score = sklearn_mape(y_test, y_pred) * 100
rmse = np.sqrt(np.mean((y_test.values - y_pred)**2))

print(f'LightGBM Forecaster Results:')
print(f'  MAPE: {mape_score:.2f}%')
print(f'  RMSE: {rmse:.4f}')

# Feature importance
fi = pd.Series(lgbm.feature_importances_, index=X.columns)
fi_sorted = fi.sort_values(ascending=False)

print(f'\nTop 10 features by importance:')
for feat, imp in fi_sorted.head(10).items():
    bar = '█' * int(imp / fi_sorted.max() * 20)
    print(f'  {feat:25s} {bar} ({imp})')
